# Pipeline V8 — Fase 1: Geração Local (referência)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-004).

Esta fase **não é executada** neste notebook — ela já foi executada com o pipeline V7 DAC.
O notebook de geração original está em:

```
notebooks/jogo_pontinhos/Geracao_Amostras_v7_adaptativo.ipynb
```

Os dados produzidos estão em:

```
dados/profundidade_minimax_11_v7_adaptativo/
```

**Saída desta fase** (schema v2-f2, já existente):

| Campo | Shape | Dtype | Descrição |
|---|---|---|---|
| `estados` | `(N, 9, 7)` | int8 | Matriz crua `{0,1,8,9}` |
| `qtd_tracos` | `(N,)` | int8 | Traços aplicados (1–30) |
| `score_jogada` | `(N, 31)` | float32 | Q-values Minimax adaptativo |
| `depth_jogada` | `(N,)` | int8 | Profundidade usada neste estado |
| `depth_geracao` | `(N,)` | int8 | Profundidade do estado anterior |
| `melhor_jogada` | `(N,)` | U5 | Argmax Minimax p=11 (Fase 2) |
| `score_melhor_jogada` | `(N, 31)` | float32 | Q-values Minimax p=11 |
| `depth_melhor_jogada` | `(N,)` | int8 | Profundidade Fase 2 (=11) |
| `labels_canonicos` | `(31,)` | U5 | Nomes canônicos dos 31 slots |

**Próxima etapa**: `fase2_enriquecimento_local.ipynb`.

In [ ]:
# Auditoria rapida: confirmar que os 152 NPZs v2-f2 estao presentes.
import os
import glob
import numpy as np

ROOT = os.environ.get('ARENA_SAGAZ_BACKEND', os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
DIR_NPZ = os.path.join(ROOT, 'dados', 'profundidade_minimax_11_v7_adaptativo')

arquivos_originais = sorted([
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if not any(f.endswith(s + '.npz') for s in ('_refH', '_refV', '_r180'))
])

print(f'NPZs originais encontrados: {len(arquivos_originais)}')

# Verifica o primeiro arquivo para confirmar schema v2-f2.
if arquivos_originais:
    d = np.load(arquivos_originais[0], allow_pickle=False)
    campos_v2f2 = {'estados', 'qtd_tracos', 'score_jogada', 'depth_jogada',
                   'depth_geracao', 'melhor_jogada', 'score_melhor_jogada',
                   'depth_melhor_jogada', 'labels_canonicos'}
    presentes = set(d.files)
    faltando = campos_v2f2 - presentes
    print(f'Schema v2-f2 OK: {not bool(faltando)}')
    if faltando:
        print(f'  Campos ausentes: {faltando}')
    print(f'  estados.shape: {d["estados"].shape}')
    print(f'  Campos presentes: {sorted(presentes)}')
    tem_canais = 'canais' in presentes
    if tem_canais:
        n_canais = d['canais'].shape[-1]
        print(f'  [ENRIQUECIDO] canais.shape: {d["canais"].shape} — {n_canais} canais')
    else:
        print('  [NAO ENRIQUECIDO] canais ausente — executar fase2_enriquecimento_local.ipynb')